# Training base expert on vanilla OGBench environment using BC (antgiant)

In [1]:
# import random
# import torch
# import os
# import math

# import matplotlib.pyplot as plt

# from collections import defaultdict

# from causal_gym import AntMazePCH
# from causal_rl.algo.imitation.imitate import *
# from causal_rl.algo.imitation.finetune import *

In [2]:
# os.environ['CUDA_VISIBLE_DEVICES'] = '2'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device

In [3]:
# num_steps = 1000
# seed = 0
# hidden_dims = {'W'}

# random.seed(seed)
# torch.manual_seed(seed)

In [4]:
# env = AntMazePCH(num_steps=num_steps, custom_hidden=hidden_dims, expert_mode=True, seed=seed, env_id='antmaze-giant-navigate-singletask-task1-v0')
# train_eps = env.expert.num_eps
# train_eps

In [5]:
# X = {f'X{t}' for t in range(num_steps)}
# Y = f'Y{num_steps}'
# obs_prefix = env.env.observed_unobserved_vars[0]

In [6]:
# Z_sets = {}
# for Xi in X:
#     i = int(Xi[1:])
#     cond = set()

#     for j in range(i+1):
#         cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

#     for j in range(i):
#         cond.add(f'X{j}')

#     Z_sets[Xi] = cond

# Z_sets['X1']

In [7]:
# records = collect_expert_trajectories(
#     env,
#     num_episodes=train_eps,
#     max_steps=num_steps,
#     seed=seed,
#     show_progress=True
# )

In [8]:
# hidden_size = 256
# lr = 3e-4
# batch_size = 2048
# patience = 20
# lookback = 10
# num_blocks = 4
# epochs = 300
# dropout = 0.0

# dims = {
#     'P': 3,
#     'O': 4,
#     'A': 8,
#     'L': 3,
#     'T': 3,
#     'J': 8,
#     # 'W': 2,
#     'X': 8,
# }

In [9]:
# model, slots, Z_trim = train_single_policy_long_horizon(
#     records,
#     Z_sets,
#     dims=dims,
#     epochs=epochs,
#     include_vars=obs_prefix,
#     lookback=lookback,
#     continuous=True,
#     num_actions = env.action_space.shape[0],
#     hidden_dim=hidden_size,
#     num_blocks=num_blocks,
#     dropout=dropout,
#     lr=lr,
#     batch_size=batch_size,
#     patience=patience,
#     device=device,
#     seed=seed,
#     action_bounds=(env.action_space.low, env.action_space.high)
# )

# policy = shared_policy_fn_long_horizon(model, slots, Z_trim, continuous=True, device=device)
# policies = make_shared_policy_dict(policy)

In [10]:
# expert_episode_rewards = defaultdict(float)
# for rec in records:
#     ep = rec['episode']
#     expert_episode_rewards[ep] += float(rec['reward'])

# num_eps = len(expert_episode_rewards)
# expert_rewards = [expert_episode_rewards[e] for e in range(num_eps)]

# num_eval_eps = 20

# policy_records = collect_imitator_trajectories(
#     env=env,
#     policies=policies,
#     num_episodes=num_eval_eps,
#     max_steps=num_steps,
#     hidden_dims=hidden_dims,
#     show_progress=True
# )

# policy_episode_rewards = defaultdict(float)
# for rec in policy_records:
#     ep = rec['episode']
#     policy_episode_rewards[ep] += float(rec['reward'])

# policy_rewards = [policy_episode_rewards[e] for e in range(num_eval_eps)]

# sum(expert_rewards)/num_eps, sum(policy_rewards)/num_eval_eps

In [11]:
# # save model for fine-tuning
# import os
# import torch

# SAVE_DIR = '/home/et2842/causal/causalrl/models'
# os.makedirs(SAVE_DIR, exist_ok=True)
# MODEL_PATH = os.path.join(SAVE_DIR, 'antmaze_giant_expert.pt')

# checkpoint = {
#     "state_dict": model.state_dict(),
#     "slots": slots,
#     "Z_trim": Z_trim,
#     "dims": dims,
#     "lookback": lookback,
#     "continuous": True,
#     "num_actions": env.action_space.shape[0],
#     "hidden_dim": hidden_size,
#     "num_blocks": num_blocks,
#     "dropout": 0.0,
#     "layernorm": True,
#     "final_tanh": True,
#     "action_bounds_low": env.action_space.low,
#     "action_bounds_high": env.action_space.high,
#     "input_dim": int(model.hidden.in_features),
# }

# torch.save(checkpoint, MODEL_PATH)
# print("Saved expert to:", MODEL_PATH)

# Fine-tuning expert on AntMaze Giant (antgiant)

In [12]:
import random
import torch
import os
import math

import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [13]:
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
# load model
MODEL_PATH = "/home/et2842/causal/causalrl/models/antmaze_giant_expert.pt"
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Rebuild the model with the same architecture
action_bounds = (checkpoint['action_bounds_low'], checkpoint['action_bounds_high'])

pretrained_actor = ContinuousPolicyNN(
    input_dim=checkpoint['input_dim'],
    action_dim=checkpoint['num_actions'],
    hidden_dim=checkpoint['hidden_dim'],
    num_blocks=checkpoint['num_blocks'],
    dropout=checkpoint['dropout'],
    layernorm=checkpoint['layernorm'],
    final_tanh=checkpoint['final_tanh'],
    action_bounds=action_bounds,
).to(device)

pretrained_actor.load_state_dict(checkpoint['state_dict'])
# pretrained_actor.eval()
pretrained_actor.train()

slots = checkpoint['slots']
Z_trim = checkpoint['Z_trim']
dims = checkpoint['dims']
lookback = checkpoint['lookback']

state_dim = checkpoint['input_dim']
state_dim, lookback

/tmp/ipykernel_3544158/2386117712.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_PATH, map_location=device)


(440, 10)

In [15]:
num_steps = 1000
rl_seed_pretrain = 2014
rl_seed = 90210
hidden_dims = set() # {'W'}

env_pretrain = AntMazePCH(env_id='antmaze-giant-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, seed=rl_seed_pretrain)
env_train = AntMazePCH(env_id='antmaze-giant-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, seed=rl_seed)
action_dim = env_train.env.action_space.shape[0]
action_dim

8

In [16]:
# reward shaping
def make_dense_distance_reward(env, use_delta=True, c=1.0, success_bonus=50.0, success_radius=5.0):
        goal_xy = env.env._goal_xy
    
        def reward_fn(obs, reward_env):
            t = len(obs['P']) - 1
    
            P_curr = obs['P'][t]
            curr_xy = np.array(P_curr[:2], dtype=np.float64)
            dist_curr = np.linalg.norm(curr_xy - goal_xy)
    
            r = 0.0
            if use_delta:
                if t == 0:
                    r = 0.0
                else:
                    P_prev = obs['P'][t - 1]
                    prev_xy = np.array(P_prev[:2], dtype=np.float64)
                    dist_prev = np.linalg.norm(prev_xy - goal_xy)
                    r = float(c * (dist_prev - dist_curr))
            else:
                r = float(-c * dist_curr)

            if dist_curr <= success_radius:
                r += success_bonus
    
            return r
    
        return reward_fn

reward_fn = make_dense_distance_reward(env_train)

In [17]:
config = OnlineRLConfig(
    total_env_steps=1_000_000,
    start_steps=20_000,
    max_episode_steps=num_steps,
    batch_size=512,
    gamma=0.99,
    tau=0.005,
    policy_delay=2,
    actor_lr=3e-4,
    critic_lr=3e-4,
    noise_std=0.15,
    hidden_dim_q=256,
    target_policy_noise=0.2,
    target_noise_clip=0.3,
    actor_warmup_steps=150_000,
    bc_reg_lambda=0.01,
    max_grad_norm=1.0
)

In [18]:
# pretrain critics offline
replay_buffer, q1, q2, target_q1, target_q2 = pretrain_critics_offline(
    env=env_pretrain,
    pretrained_actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    num_pretrain_steps=300_000,
    pretrain_updates=150_000,
    seed=rl_seed_pretrain,
    reward_shaping_fn=reward_fn
)

In [19]:
def callback(stats: dict):
    if stats['episode'] % 1 == 0:
        print(
            f'[Episode {stats["episode"]}] '
            f'steps={stats["env_steps"]}, '
            f'return={stats["return"]:.2f}, '
            f'len={stats["length"]}, '
            f'buffer={stats["buffer_size"]}'
        )

In [20]:
fine_tuned_policy, logs = td3_fine_tune_actor(
    env=env_train,
    actor=pretrained_actor,
    Z_trim=Z_trim,
    slots=slots,
    state_dim=state_dim,
    action_dim=action_dim,
    config=config,
    device=device,
    seed=rl_seed,
    log_callback=callback,
    replay_buffer=replay_buffer,
    initial_q1=q1,
    initial_q2=q2,
    initial_target_q1=target_q1,
    initial_target_q2=target_q2,
    reward_shaping_fn=reward_fn
)

ft_pi = shared_policy_fn_long_horizon(fine_tuned_policy, slots, Z_trim, continuous=True, device=device)
ft_policies = make_shared_policy_dict(ft_pi)

[Episode 1] steps=1000, return=11.88, len=1000, buffer=301000


[Episode 2] steps=2000, return=7.20, len=1000, buffer=302000


[Episode 3] steps=3000, return=16.08, len=1000, buffer=303000


[Episode 4] steps=4000, return=19.20, len=1000, buffer=304000


[Episode 5] steps=5000, return=17.35, len=1000, buffer=305000


...

[Episode 996] steps=996000, return=7.76, len=1000, buffer=1000000


[Episode 997] steps=997000, return=9.71, len=1000, buffer=1000000


[Episode 998] steps=998000, return=12.32, len=1000, buffer=1000000


[Episode 999] steps=999000, return=5.43, len=1000, buffer=1000000


[Episode 1000] steps=1000000, return=19.25, len=1000, buffer=1000000


In [21]:
expert_env = AntMazePCH(env_id='antmaze-giant-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True)

In [22]:
# save expert
import os
import torch

SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'antmaze_giant_expert_finetuned.pt')

checkpoint = {
    "state_dict": fine_tuned_policy.state_dict(),
    "slots": slots,
    "Z_trim": Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": env_train.action_space.shape[0],
    "hidden_dim": config.hidden_dim_q,
    "num_blocks": checkpoint['num_blocks'],
    "dropout": 0.0,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": env_train.action_space.low,
    "action_bounds_high": env_train.action_space.high,
    "input_dim": int(fine_tuned_policy.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print("Saved expert to:", MODEL_PATH)

Saved expert to: /home/et2842/causal/causalrl/models/antmaze_giant_expert_finetuned.pt


In [23]:
num_eval_eps = 1000

expert_returns = collect_imitator_trajectories(
    env=expert_env,
    policies=ft_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True
)

Starting episode 1/1000...


  Episode 1 ended at step 1000 (terminated: False, truncated: True).
Starting episode 2/1000...


  Episode 2 ended at step 1000 (terminated: False, truncated: True).
Starting episode 3/1000...


  Episode 3 ended at step 1000 (terminated: False, truncated: True).
Starting episode 4/1000...


  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/1000...


...

  Episode 996 ended at step 1000 (terminated: False, truncated: True).
Starting episode 997/1000...


  Episode 997 ended at step 1000 (terminated: False, truncated: True).
Starting episode 998/1000...


  Episode 998 ended at step 1000 (terminated: False, truncated: True).
Starting episode 999/1000...


  Episode 999 ended at step 1000 (terminated: False, truncated: True).
Starting episode 1000/1000...


  Episode 1000 ended at step 1000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


In [24]:
expert_episode_rewards = defaultdict(float)
for rec in expert_returns:
    ep = rec['episode']
    expert_episode_rewards[ep] += float(rec['reward'])

expert_rewards = [expert_episode_rewards[e] for e in range(num_eval_eps)]
sum(expert_rewards) / num_eval_eps

-560.6010726459525

In [25]:
mean_reward = np.mean(expert_rewards)
std_reward = np.std(expert_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] \u00b1 Std[Y] = {mean_reward:.4f} \u00b1 {std_reward:.4f}")

E[Y]          = -560.6011
Std[Y]        = 175.6169
E[Y] ± Std[Y] = -560.6011 ± 175.6169


In [26]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in expert_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

Success rate   = 0.00% (0/1000 episodes)
Std error      = 0.00%


In [27]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")

No episodes were solved.
